BÀI THỰC HÀNH SỐ 1 - SPARKSQL: KHÁM PHÁ, XỬ LÝ VÀ PHÂN TÍCH DỮ LIỆU NYC TAXI
Tập dữ liệu: NYC Yellow Taxi 2023 (Tệp Parquet các tháng) & Taxi Zone Lookup (Tệp CSV).
Mục tiêu: Xây dựng quy trình xử lý dữ liệu (Data Pipeline) từ dữ liệu thô đến báo cáo phân tích theo chuẩn kỹ thuật Data Engineering chuyên nghiệp.

Các thuộc tính chính trong bộ dữ liệu:
1.	VendorID: Mã chỉ định nhà cung cấp dịch vụ taxi (Creative Mobile Technologies hoặc VeriFone Inc.).
2.	tpep_pickup_datetime: Thời gian khi đồng hồ tính tiền bắt đầu.
3.	tpep_dropoff_datetime: Thời gian khi đồng hồ tính tiền dừng.
4.	Passenger_count: Số lượng hành khách trong xe (giá trị do tài xế nhập).
5.	Trip_distance: Khoảng cách chuyến đi tính bằng dặm.
6.	Pickup_longitude và Pickup_latitude: Vị trí đón khách (kinh độ và vĩ độ).
7.	RateCodeID: Mã loại giá áp dụng cho chuyến đi (ví dụ: giá chuẩn, giá sân bay JFK, giá thỏa thuận…).
8.	Store_and_fwd_flag: Cờ cho biết liệu chuyến đi có bị tạm lưu trong bộ nhớ xe do mất kết nối với máy chủ hay không.
9.	Dropoff_longitude và Dropoff_latitude: Vị trí trả khách.
10.	Payment_type: Hình thức thanh toán (tiền mặt, thẻ tín dụng, không tính phí…).
11.	Fare_amount: Cước phí chuyến đi tính theo thời gian và khoảng cách.
12.	Extra: Các phụ phí như phí giờ cao điểm, phí qua đêm.
13.	MTA_tax: Thuế MTA tự động áp dụng cho chuyến đi.
14.	Improvement_surcharge: Phụ phí cải tiến áp dụng từ năm 2015.
15.	Tip_amount: Tiền tip (chỉ tính với thẻ tín dụng, không tính tiền tip bằng tiền mặt).
16.	Tolls_amount: Tổng tiền phí cầu đường phải trả trong chuyến đi.
17.	Total_amount: Tổng số tiền hành khách phải trả (không bao gồm tiền tip bằng tiền mặt).


BƯỚC 1: TẢI DỮ LIỆU VÀ CHUẨN BỊ MÔI TRƯỜNG
Học viên chạy đoạn mã Python sau để tự động tải các tệp dữ liệu về thư mục local ./nyc_data:

In [ ]:
import os
import urllib.request

# Tạo thư mục lưu trữ
os.makedirs("./nyc_data", exist_ok=True)

# Danh sách URL dữ liệu taxi NYC năm 2023 và bảng tra cứu
urls = {
    "yellow_tripdata_2023-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet",
    "yellow_tripdata_2023-02.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet",
    "taxi_zone_lookup.csv": "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
}

for file_name, url in urls.items():
    file_path = os.path.join("./nyc_data", file_name)
    if not os.path.exists(file_path):
        print(f"Đang tải {file_name}...")
        urllib.request.urlretrieve(url, file_path)
        print(f"Đã tải: {file_name}")
    else:
        print(f"⚡ File đã tồn tại: {file_name}")

BƯỚC 2: KHỞI TẠO SPARK SESSION VỚI CẤU HÌNH ỔN ĐỊNH


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC_Taxi_BigData_Lab") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "16") \
    .getOrCreate()

print("Spark Session đã được khởi tạo thành công!")

BƯỚC 3: NẠP DỮ LIỆU & CHUẨN HÓA SCHEMA KHÔNG ĐỒNG NHẤT
Dữ liệu thực tế giữa các tháng thường bị lệch kiểu (INT vs BIGINT, DOUBLE vs BIGINT) và tên cột (Airport_fee vs airport_fee). Chúng ta áp dụng giải pháp ép kiểu tối giản dựa trên Schema khuôn mẫu của Tháng 1.

In [ ]:
from pyspark.sql import functions as F
# 1. Nạp tệp Tháng 1 (Làm khuôn mẫu Schema) và Tháng 2
df1 = spark.read.parquet("./nyc_data/yellow_tripdata_2023-01.parquet")
df2 = spark.read.parquet("./nyc_data/yellow_tripdata_2023-02.parquet")

# 2. Chuẩn hóa tên cột bị lệch chữ hoa/thường ở df2
df2 = df2.withColumnRenamed("Airport_fee", "airport_fee")

# 3. Ép toàn bộ kiểu dữ liệu của df2 theo Schema khuôn mẫu của df1
df2 = df2.select([F.col(col.name).cast(col.dataType) for col in df1.schema])

# 4. Hợp nhất hai DataFrame
df_raw = df1.union(df2)

# 5. Nạp bảng danh mục khu vực (CSV)
df_zones = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("./nyc_data/taxi_zone_lookup.csv")

total_records = df_raw.count()
print(f"Tổng số bản ghi nạp vào: {total_records:,} dòng")

BƯỚC 4: KHÁM PHÁ DỮ LIỆU CHUYÊN SÂU (EDA) & THỐNG KÊ HÌNH DẠNG
4.1 Thống kê mô tả và kiểm tra giá trị khuyết thiếu (NULL)

In [ ]:
# 1. Thống kê phân vị cơ bản
numeric_cols = ["passenger_count", "trip_distance", "fare_amount", "tip_amount", "total_amount"]
print("=== BÁO CÁO THỐNG KÊ MÔ TẢ ===")
df_raw.select(numeric_cols).summary("count", "mean", "stddev", "min", "50%", "max").show()

# 2. Kiểm tra số lượng NULL trên tất cả các cột bằng List Comprehension
null_counts = df_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_raw.columns
])
print("=== SỐ LƯỢNG GIÁ TRỊ NULL THEO CỘT ===")
null_counts.show(vertical=True)

4.2 Thống kê Độ lệch (Skewness), Độ nhọn (Kurtosis) & Tương quan (Pearson Matrix)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Tính toán các cờ thời gian tạm thời để phân tích
df_eda = df_raw \
    .withColumn("trip_duration_min", F.round((F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60, 2))

# 1. Tính Skewness & Kurtosis
stat_cols = ["trip_distance", "fare_amount", "tip_amount", "total_amount", "trip_duration_min"]
skew_kurt_exprs = []
for c in stat_cols:
    skew_kurt_exprs.append(F.round(F.skewness(c), 3).alias(f"{c}_skewness"))
    skew_kurt_exprs.append(F.round(F.kurtosis(c), 3).alias(f"{c}_kurtosis"))

print("=== THỐNG KÊ SKEWNESS VÀ KURTOSIS ===")
df_eda.select(skew_kurt_exprs).show(vertical=True)

# 2. Tính Ma trận Tương quan Pearson
corr_matrix = []
for c1 in stat_cols:
    row = []
    for c2 in stat_cols:
        val = df_eda.select(F.corr(c1, c2)).collect()[0][0]
        row.append(round(val, 3) if val is not None else 0.0)
    corr_matrix.append(row)

df_corr = pd.DataFrame(corr_matrix, index=stat_cols, columns=stat_cols)

# 3. Trực quan hóa Heatmap Tương quan bằng Matplotlib thuần
plt.figure(figsize=(8, 6))
im = plt.imshow(df_corr.values, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(im, label="Hệ số tương quan")
plt.xticks(np.arange(len(stat_cols)), stat_cols, rotation=30, ha="right")
plt.yticks(np.arange(len(stat_cols)), stat_cols)

for i in range(len(stat_cols)):
    for j in range(len(stat_cols)):
        color = "white" if abs(df_corr.values[i, j]) > 0.6 else "black"
        plt.text(j, i, f"{df_corr.values[i, j]:.2f}", ha="center", va="center", color=color, fontweight="bold")

plt.title("Ma Trận Tương Quan Pearson Dữ Liệu Taxi NYC", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

Bài tập 1 (Phân tích EDA):
Dựa vào kết quả thu được, hãy lý giải:
+ Tại sao trip_distance có Skewness và Kurtosis tăng vọt? Điều này báo hiệu điều gì về chất lượng dữ liệu?
+ Dựa vào Ma trận Tương quan, cặp biến nào có tương quan tuyến tính mạnh nhất với total_amount?

BƯỚC 5: PHÁT HIỆN DỮ LIỆU BẤT THƯỜNG (ANOMALY DETECTION) & LÀM SẠCH
5.1 Gán cờ báo lỗi logic (Anomaly Flagging) & Thống kê

In [ ]:
# Gán cờ báo hiệu bất thường dạng Array
df_flagged = df_eda \
    .withColumn("speed_mph", F.round(F.col("trip_distance") / (F.col("trip_duration_min") / 60), 2)) \
    .withColumn("anomaly_reasons", F.array(
        F.when(F.col("passenger_count") <= 0, "INVALID_PASSENGER"),
        F.when(F.col("trip_distance") <= 0, "ZERO_OR_NEGATIVE_DISTANCE"),
        F.when(F.col("total_amount") <= 0, "ZERO_OR_NEGATIVE_FARE"),
        F.when(F.col("trip_duration_min") <= 0, "INVALID_DURATION"),
        F.when(F.col("trip_duration_min") > 1440, "EXCESSIVE_DURATION"),
        F.when(F.col("speed_mph") > 100, "EXCESSIVE_SPEED"),
        F.when(F.col("PULocationID").isin(264, 265) | F.col("DOLocationID").isin(264, 265), "UNKNOWN_LOCATION")
    )) \
    .withColumn("anomaly_reasons", F.filter("anomaly_reasons", lambda x: x.isNotNull()))

# Thống kê chi tiết các dạng lỗi
print("=== BÁO CÁO TỔNG HỢP CÁC DẠNG BẤT THƯỜNG ===")
df_flagged \
    .filter(F.size("anomaly_reasons") > 0) \
    .select(F.explode("anomaly_reasons").alias("anomaly_type")) \
    .groupBy("anomaly_type") \
    .agg(F.count("*").alias("error_count")) \
    .withColumn("pct_of_total", F.round((F.col("error_count") / total_records) * 100, 2)) \
    .orderBy(F.desc("error_count")) \
    .show(truncate=False)

5.2 Xử lý Outlier theo Phương pháp Phân vị P99.9 & Lọc dữ liệu sạch

In [ ]:
# Tính ngưỡng P99.9 cho quãng đường di chuyển
p999_dist = df_raw.approxQuantile("trip_distance", [0.999], 0.01)[0]
print(f"Ngưỡng Outlier Quãng đường (P99.9): {p999_dist:.2f} miles")

# Lọc tập dữ liệu sạch hoàn toàn
df_clean = df_flagged \
    .filter(
        (F.size("anomaly_reasons") == 0) & 
        (F.col("trip_distance") <= p999_dist)
    ) \
    .withColumn("pickup_date", F.col("tpep_pickup_datetime").cast("date")) \
    .drop("anomaly_reasons")

clean_records = df_clean.count()
print(f"Dữ liệu sạch còn lại: {clean_records:,} dòng (Đã loại bỏ {total_records - clean_records:,} dòng lỗi)")

Lưu lại dữ liệu đã làm sạch

In [9]:
output_path = "./nyc_data/yellow_tripdata_2023-01-02-parquet"

df_clean.write \
    .mode("overwrite") \
    .parquet(output_path)

print(f"Đã ghi tập dữ liệu sạch thành công vào: {output_path}")

Đã ghi tập dữ liệu sạch thành công vào: ./nyc_data/yellow_tripdata_2023-01-02-parquet


Bài tập 2 (Anomaly Detection):
Viết câu lệnh đếm số lượng chuyến đi bị lỗi thời gian ngược (thời gian trả khách tpep_dropoff_datetime xảy ra trước thời gian đón khách tpep_pickup_datetime). Tính tổng thất thu tiền cước (total_amount) từ các chuyến đi bị lỗi này.

BƯỚC 6: MULTI-JOIN, AGGREGATION & XOAY BẢNG DỮ LIỆU (PIVOT)

6.1 Multi-JOIN & Lọc bỏ giá trị không xác định

In [ ]:
# Join với bảng danh mục Zone
df_joined = df_clean \
    .join(df_zones, df_clean.PULocationID == df_zones.LocationID, "left") \
    .withColumnRenamed("Borough", "pickup_borough") \
    .withColumnRenamed("Zone", "pickup_zone") \
    .drop("LocationID")

# Báo cáo tổng doanh thu theo Quận (Loại bỏ hoàn toàn 'Unknown', 'N/A' và NULL)
df_borough_summary = df_joined \
    .filter(
        F.col("pickup_borough").isNotNull() & 
        (~F.col("pickup_borough").isin("Unknown", "N/A"))
    ) \
    .groupBy("pickup_borough") \
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue")
    ) \
    .orderBy(F.desc("total_revenue"))

df_borough_summary.show()

6.2 Bảng Xoay Ma trận (Pivot Table Optimization)
Thực hiện Pivot tạo ma trận doanh thu giữa các Quận (Hàng) và Thứ trong tuần (Cột). Truyền danh sách cột cố định để tối ưu hóa hiệu năng Spark.

In [ ]:
# Thêm cột viết tắt ngày trong tuần
df_joined_day = df_joined \
    .filter(F.col("pickup_borough").isNotNull() & (~F.col("pickup_borough").isin("Unknown", "N/A"))) \
    .withColumn("day_of_week", F.date_format("pickup_date", "E"))

pivot_days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

df_pivot_revenue = df_joined_day \
    .groupBy("pickup_borough") \
    .pivot("day_of_week", pivot_days) \
    .agg(F.round(F.sum("total_amount"), 2)) \
    .fillna(0)

print("=== MA TRẬN DOANH THU THEO QUẬN VÀ NGÀY TRONG TUẦN (PIVOT) ===")
df_pivot_revenue.show(truncate=False)

Bài tập 3 (Pivot Table):
Hãy tạo ma trận Pivot đếm Số lượng chuyến đi (COUNT(*)) giữa:
Hàng: Các Quận đón khách (pickup_borough).
Cột: Phương thức thanh toán (payment_type). Chỉ xét phương thức 1 (Thẻ) và 2 (Tiền mặt).

BƯỚC 7: TRỰC QUAN HÓA DỮ LIỆU ĐA CHIỀU 
Sử dụng kỹ thuật lấy mẫu (Sampling 1%) để vẽ các biểu đồ phân tích chuyên sâu mà không làm cạn kiệt bộ nhớ RAM.

In [ ]:
# Lấy mẫu 1% dữ liệu phục vụ trực quan hóa
pdf_sample = df_joined \
    .filter(F.col("pickup_borough").isNotNull() & (~F.col("pickup_borough").isin("Unknown", "N/A"))) \
    .sample(fraction=0.01, seed=42) \
    .toPandas()

# ------------------------------------------------------------------------------
# 7.1 Biểu đồ Cột: Doanh thu theo Quận
# ------------------------------------------------------------------------------
pdf_borough = df_borough_summary.toPandas()
plt.figure(figsize=(9, 4))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(pdf_borough)))
plt.bar(pdf_borough["pickup_borough"], pdf_borough["total_revenue"], color=colors)
plt.title("Tổng Doanh Thu Taxi NYC Phân Theo Quận Đón Khách", fontsize=11, fontweight="bold")
plt.xlabel("Quận (Borough)"); plt.ylabel("Doanh thu (USD)")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout(); plt.show()

# ------------------------------------------------------------------------------
# 7.2 Biểu đồ Scatter: Quãng đường vs Tổng tiền (Tương quan)
# ------------------------------------------------------------------------------
pdf_scatter = pdf_sample[pdf_sample['total_amount'] < 100]
plt.figure(figsize=(9, 4))
plt.scatter(pdf_scatter["trip_distance"], pdf_scatter["total_amount"], alpha=0.3, color="#2b5c8f", s=15)
plt.title("Mối Tương Quan Quãng Đường - Tổng Tiền Taxi NYC (Lấy mẫu 1%)", fontsize=11, fontweight="bold")
plt.xlabel("Quãng đường (dặm)"); plt.ylabel("Tổng tiền (USD)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout(); plt.show()

# ------------------------------------------------------------------------------
# 7.3 Biểu đồ Boxplot Subplots: Phân bố Quãng đường (Toàn cảnh vs Zoom-in)
# ------------------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Subplot 1: Có Outliers (< 50 miles)
ax1.boxplot(pdf_sample[pdf_sample["trip_distance"] < 50]["trip_distance"], patch_artist=True, showfliers=True,
            flierprops=dict(marker='o', markerfacecolor='red', alpha=0.2, markersize=3))
ax1.set_title("Phân Bố Quãng Đường (Có Outliers < 50 mi)", fontsize=10, fontweight="bold")
ax1.set_ylabel("Dặm (miles)"); ax1.grid(axis='y', linestyle='--')

# Subplot 2: Zoom-in (Ẩn Outliers & Đánh dấu chỉ số)
bp2 = ax2.boxplot(pdf_sample["trip_distance"], patch_artist=True, showfliers=False)
bp2['boxes'][0].set_facecolor('#3182bd')

q1 = np.percentile(pdf_sample["trip_distance"], 25)
med = np.median(pdf_sample["trip_distance"])
q3 = np.percentile(pdf_sample["trip_distance"], 75)

ax2.text(1.08, q1, f"Q1: {q1:.2f} mi", color='darkblue')
ax2.text(1.08, med, f"Median: {med:.2f} mi", color='red', fontweight='bold')
ax2.text(1.08, q3, f"Q3: {q3:.2f} mi", color='darkblue')
ax2.set_title("Phân Bố Quãng Đường Zoom-In (Ẩn Outliers)", fontsize=10, fontweight="bold")
ax2.set_ylabel("Dặm (miles)"); ax2.grid(axis='y', linestyle='--')

plt.tight_layout(); plt.show()

BƯỚC 8: PHÂN TÍCH NÂNG CAO VỚI WINDOW FUNCTIONS & CẤU TRÚC PHỨC TẠP
8.1 Hàm Cửa sổ (Window Functions)
Tính doanh thu tích lũy cộng dồn (Running Total) và so sánh chênh lệch doanh thu với ngày liền trước (Lag) theo từng Quận.

In [ ]:
from pyspark.sql.window import Window

# Gom nhóm doanh thu theo Quận và Ngày
df_daily_borough = df_joined \
    .filter(F.col("pickup_borough").isNotNull() & (~F.col("pickup_borough").isin("Unknown", "N/A"))) \
    .groupBy("pickup_borough", "pickup_date") \
    .agg(F.round(F.sum("total_amount"), 2).alias("daily_revenue"))

# Định nghĩa cửa sổ phân vùng theo Quận và sắp xếp theo Ngày
window_spec = Window.partitionBy("pickup_borough").orderBy("pickup_date")

df_windowed = df_daily_borough \
    .withColumn("running_total_revenue", F.round(F.sum("daily_revenue").over(window_spec), 2)) \
    .withColumn("prev_day_revenue", F.lag("daily_revenue", 1).over(window_spec)) \
    .withColumn("revenue_diff", F.round(F.col("daily_revenue") - F.col("prev_day_revenue"), 2))

print("=== PHÂN TÍCH DOANH THU CỘNG DỒN VÀ CHÊNH LỆCH NGÀY (WINDOWING) ===")
df_windowed.filter(F.col("pickup_borough") == "Manhattan").orderBy("pickup_date").show(10)

8.2 Gom nhóm nâng cao với Struct & Array (Entity Profiling)

In [ ]:
# Tạo hồ sơ chi tiết các chuyến đi cước phí cao (> $50) gom nhóm theo Quận dưới dạng Mảng các Struct
df_profile = df_joined \
    .filter(F.col("total_amount") > 50) \
    .groupBy("pickup_borough") \
    .agg(
        F.collect_list(
            F.struct("tpep_pickup_datetime", "trip_distance", "total_amount")
        ).alias("high_value_trips")
    ) \
    .withColumn("total_high_value_count", F.size("high_value_trips"))

df_profile.select("pickup_borough", "total_high_value_count").show()

Bài tập 4 (Window Functions):
Sử dụng Window.partitionBy("pickup_borough").orderBy(F.desc("daily_revenue")) và hàm F.row_number(), hãy tìm Top 3 ngày có doanh thu cao nhất của từng Quận.

BƯỚC 9: TRUY VẤN SPARKSQL & XUẤT DỮ LIỆU TỐI ƯU (PARTITIONING)

9.1 Đăng ký Temporary View và Thực thi SparkSQL

In [ ]:
# Đăng ký DataFrame thành bảng tạm trong Spark catalog
df_joined.createOrReplaceTempView("vw_nyc_taxi")

# Thực hiện truy vấn SparkSQL phức tạp
spark_sql_query = """
    SELECT 
        pickup_borough,
        COUNT(*) AS total_trips,
        ROUND(AVG(trip_distance), 2) AS avg_distance,
        ROUND(AVG(total_amount), 2) AS avg_fare,
        ROUND(SUM(tip_amount) / SUM(total_amount) * 100, 2) AS tip_percentage
    FROM vw_nyc_taxi
    WHERE pickup_borough NOT IN ('Unknown', 'N/A') AND pickup_borough IS NOT NULL
    GROUP BY pickup_borough
    HAVING total_trips > 1000
    ORDER BY tip_percentage DESC
"""

print("=== KẾT QUẢ TRUY VẤN SPARKSQL ===")
spark.sql(spark_sql_query).show()

9.2 Xuất dữ liệu sạch ra định dạng Parquet theo Phân vùng (Partitioning)

In [ ]:
# Ghi dữ liệu sạch ra đĩa dưới dạng Parquet được phân vùng theo Ngày đón khách
output_path = "./nyc_data/output_clean_taxi_parquet"

df_joined.write \
    .mode("overwrite") \
    .partitionBy("pickup_date") \
    .parquet(output_path)

print(f"Đã ghi tập dữ liệu sạch thành công vào: {output_path}")

Bài tập 5: 

1: Phân tích Tuyến đường Huyết mạch theo Khung giờ
Tìm Top 5 cặp tuyến đường phổ biến nhất (`PULocationID` -> `DOLocationID` >= 5,000 chuyến). Tính thời gian di chuyển trung bình trong Giờ Cao Điểm (7h-9h & 17h-19h) so với Giờ Thấp Điểm (22h-4h).

2: Tối ưu hóa hiệu năng Partition Pruning
- Ghi `df_clean` ra đĩa dạng Parquet phân vùng theo ngày: `df_clean.write.partitionBy('pickup_date').mode('overwrite').parquet('./nyc_data_partitioned')`
- Nạp lại dữ liệu phân vùng và so sánh thời gian truy vấn ngày `'2023-01-15'` giữa `df_clean` và `df_partitioned`.